# Turn any GitHub repo into a knowledge graph Claude can query

In ~50 lines, this notebook:

1. Clones a famous open-source repo (Flask), parses it into a code knowledge graph with `kglite.code_tree`, and asks a couple of Cypher questions to prove the library works standalone.
2. Sets up a **workspace MCP server** and registers it in Claude Desktop, so going forward you can ask Claude things like *"clone tiangolo/fastapi and tell me its routes"* — the agent calls `repo_management('org/repo')`, KGLite clones + builds the graph, and follow-up `cypher_query()` calls flow against it.

*A knowledge graph here means: functions, classes, and files become nodes; calls, imports, and inheritance become edges; the result is queryable with Cypher — a SQL-shaped graph query language.*

Requirements: `pip install 'kglite[mcp]'`, a working `git` binary, and (for step 2) the Claude Desktop app installed.


In [ ]:
from pathlib import Path

from kglite.code_tree import build
from kglite.mcp_server.workspace import Workspace

REPO = "pallets/flask"  # swap for any GitHub repo — kglite handles graphs up to Wikidata scale

# Workspace owns the clones and auto-prunes any repo idle longer than
# `stale_after_days`. The MCP server in cell 4 uses the same pattern, so
# the demo stays consistent end-to-end.
ws = Workspace(root=Path.home() / "kglite-mcp-workspace", stale_after_days=7)

# repo_management clones if needed + activates. (MCP-facing schema documents
# the arg as "repo"; Python dispatcher reads "name" — pre-existing mismatch.)
ws.repo_management_tool({"name": REPO})

graph = build(str(ws.root))
print(f"Parsed {REPO}: {graph}")

# Cypher: MATCH (var:NodeType)-[:EDGE_TYPE]->(other) ... RETURN ...
# Full reference: https://kglite.readthedocs.io (CYPHER.md)
print("\nMost-called functions:")
for row in graph.cypher("""
    MATCH (c:Function)-[:CALLS]->(f:Function)
    RETURN f.qualified_name AS fn, count(c) AS callers
    ORDER BY callers DESC LIMIT 5
"""):
    print(f"  {row['callers']:>3}x {row['fn']}")

print("\nRoutes detected in the Flask repo (HANDLES edges, web-framework detector):")
for row in graph.cypher("""
    MATCH (r:Route)-[:HANDLES]->(f:Function)
    RETURN r.path AS path, f.qualified_name AS handler
    LIMIT 5
"""):
    print(f"  {row['path']:20} -> {row['handler']}")

In [ ]:
# Bridge to part 2: graph.describe() emits a compact XML schema — node
# types and their properties, edge types and direction, exploration hints
# — that fits in a system prompt. It's how the MCP server tells an LLM
# agent what's in the graph before any query is written. The agent in the
# screenshot at the bottom of this notebook saw this exact shape and used
# it to pick :Function and :CALLS for its first cypher_query() call.
xml = graph.describe()
print(xml[:700])
print(f"\n... [truncated — full schema is {len(xml):,} chars / {xml.count(chr(10))} lines]")

## 2. Hand the same capability to Claude Desktop

After the next cell you can ask Claude things like *"clone tiangolo/fastapi and show me the most-called functions"* and the agent calls `repo_management('tiangolo/fastapi')` → KGLite clones + builds the graph → `cypher_query()` answers against it.

The cell uses `kglite.mcp_server.claude_config.add_mcp` — see its docstring for `client="claude_code"` / `"vscode"` / `path=...` options. `DRY_RUN = True` prints the entry before committing.


In [ ]:
from pathlib import Path

from kglite.mcp_server import claude_config

DRY_RUN = True  # flip to False to actually write the Claude Desktop config

# Where Claude will clone repos on demand.
workspace = Path.home() / "kglite-mcp-workspace"
repos = workspace / "repos"
repos.mkdir(parents=True, exist_ok=True)

# Minimal manifest. Key fields: workspace.kind="github" enables clone-on-demand
# (Claude calls repo_management('org/repo') and kglite clones into the workspace);
# applies_to="./*" lets the manifest cover any child of the workspace parent.
# Full reference (skill packs, github_issues, per-tool overrides):
# examples/open_source_workspace_mcp.yaml
(workspace / "workspace_mcp.yaml").write_text("""\
name: Open Source Explorer
skills: true
workspace:
  kind: github
  applies_to: ./*
builtins:
  temp_cleanup: on_overview
extensions:
  csv_http_server: true
instructions: |
  Code intelligence for open-source GitHub repos via knowledge graphs.
  FIRST STEP: call repo_management('org/repo') to clone and activate.
""")

# Register in Claude Desktop. Swap client="claude_code" or "vscode" — or
# pass path="/some/custom/config.json" — to target a different client.
entry = claude_config.add_mcp(
    name="open-source",
    command="kglite-mcp-server",
    args=["--workspace", str(repos)],
    client="claude_desktop",
    overwrite=True,
    dry_run=DRY_RUN,
)

cfg_path = claude_config.default_path("claude_desktop")
print(f"{'Would write' if DRY_RUN else 'Wrote'} entry to {cfg_path}:")
print(entry)
if not DRY_RUN:
    print("\nRestart Claude Desktop, then ask:")
    print("  -> clone tiangolo/fastapi and tell me the routes it exposes")

## What you have now

After restarting Claude Desktop, the **open-source** MCP server appears in Claude's tool list. Useful first prompts:

- *"Clone pallets/flask. What are the 10 most-called functions?"*
- *"In tiangolo/fastapi, which routes are exposed and which handler owns each?"*
- *"Show me the call graph from `flask.Flask.run` two hops deep."*
- *"List the files most affected if I change `src/flask/app.py`."*

![Claude Desktop answering the first prompt against the workspace MCP server](img/claude-flask-most-called.png)

*Claude calls `repo_management('pallets/flask')` → `graph_overview()` → `cypher_query(...)` and produces the ranked list. Notice the agent's closing paragraph: it caught a real Python static-analysis subtlety (Flask's `sansio` layer means some calls resolve ambiguously to both the base and concrete class) rather than just parroting the query results.*

Under the hood Claude is calling `repo_management()` (clone + build graph), then `cypher_query()` against the resulting graph, then `read_code_source(qualified_name=...)` to fetch the source for entities the queries surface.

**Next steps:**

- Repo: [github.com/kkollsga/kglite](https://github.com/kkollsga/kglite)
- Docs: [kglite.readthedocs.io](https://kglite.readthedocs.io)
- The full annotated workspace manifest (with `github_issues`, skill packs, per-tool overrides): [`examples/open_source_workspace_mcp.yaml`](https://github.com/kkollsga/kglite/blob/main/examples/open_source_workspace_mcp.yaml)
- For a `--graph one_specific.kgl` (single-repo) setup instead of the workspace pattern, see the [MCP server guide](https://kglite.readthedocs.io/en/latest/guides/mcp-servers.html).
